# Chapitre 6 — Évaluer un modèle et mesurer l'incertitude · Notebook de travail

*Maths pour l'IA, pas à pas*, Belk Siri — notebook compagnon.

[Ouvrir dans Google Colab](https://colab.research.google.com/github/belksiri/books/blob/main/maths-pour-l-ia-pas-a-pas/notebooks/fr/ch06_a_completer.ipynb) · Page compagnon : https://belksiri.github.io/books/maths-pour-l-ia-pas-a-pas/

Les cellules à modifier portent la mention **À COMPLÉTER**. Exécutez les cellules dans l'ordre : tant qu'une réponse manque, les vérifications le signalent sans erreur. Le corrigé se trouve dans `ch06_corrige.ipynb`, sur la même page compagnon.

**Sommaire.** Données · Les calculs des leçons 6.1 à 6.6 · Exercices 6.1 à 6.6

**Le protocole, dans tout le notebook :** ajuster sur l'**entraînement**, choisir sur la **validation**, évaluer **une seule fois** sur le **test**. Pour un test A/B : comparer deux groupes **tirés au sort**, et donner l'écart **avec son intervalle de confiance**.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Couleurs du livre : une couleur par lot
LOT_E, LOT_V, LOT_T = '#2e7d32', '#b84d00', '#1565c0'   # entraînement (vert), validation (orange foncé), test (bleu)
SPAM, LEGITIME = '#d62728', '#b9b6ae'
VERSION_A, VERSION_B, ECART, SEUIL = '#757575', '#00897b', '#c2185b', '#8e24aa'   # test A/B : A gris, B sarcelle


def sigmoide(z):
    return 1 / (1 + np.exp(-np.asarray(z, dtype=float)))


def perte_moyenne(etiquettes, X, w, b):
    """Perte logarithmique moyenne (équations 5.6 et 5.7)."""
    p = np.clip(sigmoide(X @ np.asarray(w, dtype=float) + b), 1e-12, 1 - 1e-12)
    y = np.asarray(etiquettes, dtype=float)
    return float(np.mean(-(y * np.log(p) + (1 - y) * np.log(1 - p))))


NOMS_CARAC = ['liens', 'mots_suspects', 'longueur', 'majuscules', 'heure', 'pieces_jointes', 'taille', 'destinataires']


def fabriquer_messages(graine=77):
    """Messages fictifs (exercice 6.4 et leçon 6.4) : 40 groupes, campagnes de spams (groupes pairs) ou
    expéditeurs légitimes (groupes impairs). Les messages d'un groupe sont presque identiques. Seules les deux
    premières caractéristiques dépendent de l'étiquette ; les huit sont dans des unités très différentes."""
    rng = np.random.default_rng(graine)
    unites = np.array([2, 1.5, 30, 0.2, 5, 1, 50, 3])
    decalages = np.array([3, 2, 200, 1, 12, 0, 500, 10])
    lignes = []
    for g in range(40):
        spam = g % 2 == 0
        taille = rng.integers(4, 15)
        base = rng.normal(0, 1, 8)
        base[:2] += 1.0 if spam else -1.0
        for _ in range(taille):
            lignes.append([g, int(spam)] + list((base + rng.normal(0, 0.3, 8)) * unites + decalages))
    return pd.DataFrame(lignes, columns=['groupe', 'spam'] + NOMS_CARAC)


def ajuster(X, y, lam, l1=False, pas=0.1, n_pas=3000):
    """Descente de gradient sur la perte pénalisée : L2 (équation 6.2) ou L1 (équation 6.3). b n'est pas pénalisé."""
    w, b = np.zeros(X.shape[1]), 0.0
    for _ in range(n_pas):
        ecart = sigmoide(X @ w + b) - y
        gw, gb = X.T @ ecart / len(y), ecart.mean()
        if l1:   # pas de gradient, puis on rapproche chaque poids de 0 d'au plus pas × λ (sans le faire changer de signe)
            w = w - pas * gw
            w = np.sign(w) * np.maximum(np.abs(w) - pas * lam, 0)
        else:    # la dérivée de λ × somme des w² est 2λw
            w = w - pas * (gw + 2 * lam * w)
        b -= pas * gb
    return w, b

## Données

Les identifiants D6.1 à D6.5, D6.E1, D6.E2, D6.E3, D6.E5 et D5.1 sont ceux du livre et des fichiers CSV.

In [ ]:
# Données du chapitre 6, recopiées des fichiers CSV du livre (ne pas modifier)

# D6.1 — les trois lots de l'association : messages, spams, et erreurs du filtre pour un seuil
lots = pd.DataFrame([
    ('entrainement', 600, 120, 0.7, 30, 3),
    ('validation', 200, 40, 0.3, 4, 14),
    ('validation', 200, 40, 0.5, 8, 6),
    ('validation', 200, 40, 0.7, 15, 2),
    ('test', 250, 50, 0.7, 20, 4)
], columns=['lot', 'messages', 'spams', 'seuil', 'fn', 'fp'])

# D6.2 — deux découpages des mêmes messages : le lot de test, au seuil 0,7
fuite = pd.DataFrame([
    ('aleatoire', 250, 50, 25, 1, 10, 4),
    ('par_campagne', 250, 50, 0, 0, 20, 4)
], columns=['decoupage', 'messages', 'spams', 'copies_vues', 'manques_copies', 'manques_nouveaux', 'fp'])

# D6.3 — six messages nouveaux (validation de la leçon 6.3)
six = pd.DataFrame([
    ('R1', 'spam', 4, 2),
    ('R2', 'legitime', 1, 3),
    ('R3', 'spam', 3, 4),
    ('R4', 'legitime', 0, 0),
    ('R5', 'spam', 5, 3),
    ('R6', 'legitime', 2, 1)
], columns=['message', 'etiquette', 'liens', 'mots_suspects'])

# D5.1 — les douze messages du chapitre 5 (entraînement de la leçon 6.3)
douze = pd.DataFrame([
    ('M1', 'spam', 5, 4),
    ('M2', 'legitime', 2, 0),
    ('M3', 'legitime', 4, 0),
    ('M4', 'spam', 3, 3),
    ('M5', 'legitime', 1, 0),
    ('M6', 'legitime', 2, 2),
    ('M7', 'spam', 2, 3),
    ('M8', 'legitime', 2, 1),
    ('M9', 'spam', 4, 4),
    ('M10', 'legitime', 1, 1),
    ('M11', 'legitime', 0, 0),
    ('M12', 'legitime', 0, 1)
], columns=['message', 'etiquette', 'liens', 'mots_suspects'])

# D6.E2 — la médiathèque (exercice 6.2) : lots de validation et de test
mediatheque = pd.DataFrame([
    ('validation', 150, 30, 0.4, 3, 18),
    ('validation', 150, 30, 0.5, 6, 9),
    ('validation', 150, 30, 0.6, 10, 5),
    ('validation', 150, 30, 0.7, 15, 3),
    ('test', 240, 48, 0.4, 6, 26),
    ('test', 240, 48, 0.5, 10, 14),
    ('test', 240, 48, 0.6, 17, 9),
    ('test', 240, 48, 0.7, 22, 3)
], columns=['lot', 'messages', 'spams', 'seuil', 'fn', 'fp'])

# D6.4 — les trois plis de l'association (leçon 6.4) : liens (moyenne, écart-type) et pertes de validation
plis = pd.DataFrame([
    (1, 270, 54, 2.10, 1.60, 0.29, 0.30),
    (2, 264, 52, 2.40, 1.90, 0.38, 0.33),
    (3, 266, 54, 2.20, 1.70, 0.33, 0.31)
], columns=['pli', 'messages', 'spams', 'moyenne_liens', 'ecart_type_liens', 'perte_l001', 'perte_l01'])

# D6.5 — le test A/B de la librairie (leçons 6.5 et 6.6)
librairie = pd.DataFrame([
    ('A', 'lettre actuelle', 2500, 200),
    ('B', 'livraison offerte', 2400, 240)
], columns=['version', 'description', 'clients', 'commandes'])

# D6.E1 — validation croisée de la médiathèque (exercice 6.1) : perte de chaque pli pour cinq λ
plis_lambda = pd.DataFrame([
    (1, 0.001, 0.52),
    (1, 0.01, 0.44),
    (1, 0.1, 0.36),
    (1, 1, 0.35),
    (1, 10, 0.52),
    (2, 0.001, 0.40),
    (2, 0.01, 0.37),
    (2, 0.1, 0.34),
    (2, 1, 0.38),
    (2, 10, 0.55),
    (3, 0.001, 0.61),
    (3, 0.01, 0.49),
    (3, 0.1, 0.39),
    (3, 1, 0.37),
    (3, 10, 0.51)
], columns=['pli', 'lambda', 'perte'])

# D6.E3 — répartition de quatre groupes dans les plis du bénévole (exercice 6.3)
groupes_plis = pd.DataFrame([
    ('Montres', 'campagne de spams', 9, 8, 9),
    ('Colis en attente', 'campagne de spams', 5, 6, 4),
    ('Loterie', 'campagne de spams', 7, 7, 8),
    ('Lettre de la mairie', 'expéditeur légitime', 4, 3, 5)
], columns=['groupe', 'nature', 'pli_1', 'pli_2', 'pli_3'])

# D6.E5 — le test A/B du théâtre (exercice 6.5)
theatre = pd.DataFrame([
    ('A', 'lettre habituelle', 1800, 126),
    ('B', 'lettre avec vidéo', 1750, 147)
], columns=['version', 'description', 'spectateurs', 'reservations'])

## Les calculs des leçons, en Python

Ces cellules refont les calculs des leçons 6.1 à 6.6.

In [ ]:
# Leçon 6.1 — coût par message (équation 6.1), k = 3 pour l'association
k = 3
t = lots.copy()
t['coût'] = t['fn'] + k * t['fp']
t['coût par message'] = (t['coût'] / t['messages']).round(3)    # on divise par la taille DU LOT
print(t.to_string(index=False))
val = t[t['lot'] == 'validation']
choisi = val.loc[val['coût par message'].idxmin(), 'seuil']
print('seuil choisi sur la validation :', choisi)
test = t[(t['lot'] == 'test') & (t['seuil'] == choisi)].iloc[0]
print(f"test, une seule fois : coût {test['coût']} sur {test['messages']} messages = {test['coût par message']:.3f} par message")

a7 = t[t['seuil'] == 0.7]
couleurs = {'entrainement': LOT_E, 'validation': LOT_V, 'test': LOT_T}
plt.figure(figsize=(5, 2.8))
plt.bar(a7['lot'], a7['coût par message'], color=[couleurs[l] for l in a7['lot']])
plt.ylabel('Coût par message (seuil 0,7)'); plt.show()

In [ ]:
# Leçon 6.2 — une fuite de données : rappel et coût par message des deux découpages
for _, l in fuite.iterrows():
    manques = l['manques_copies'] + l['manques_nouveaux']
    nouveaux = l['spams'] - l['copies_vues']
    print(f"{l['decoupage']:>12} : rappel = {(l['spams'] - manques) / l['spams']:.2f}, "
          f"coût par message = {(manques + 3 * l['fp']) / l['messages']:.3f}, "
          f"part des spams nouveaux manqués = {l['manques_nouveaux']} / {nouveaux} = {l['manques_nouveaux'] / nouveaux:.2f}")

# Découper les campagnes, pas les messages : petite démonstration avec 40 campagnes de taille variable
rng = np.random.default_rng(6)
campagne = np.repeat(np.arange(40), rng.integers(5, 40, size=40))          # numéro de campagne de chaque spam
au_hasard = rng.random(campagne.size) < 0.25                              # 25 % des MESSAGES au test
camp_test = rng.choice(40, size=10, replace=False)                         # 25 % des CAMPAGNES au test
par_campagne = np.isin(campagne, camp_test)
for nom, test_ in (('au hasard', au_hasard), ('par campagne', par_campagne)):
    partagees = len(set(campagne[test_]) & set(campagne[~test_]))
    print(f'{nom:>12} : {test_.sum()} spams au test, {partagees} campagnes présentes à la fois au test et ailleurs')

In [ ]:
# Leçon 6.3 — surapprentissage : filtres A et B sur l'entraînement (douze messages) et la validation (six)
X12, y12 = douze[['liens', 'mots_suspects']].to_numpy(float), (douze['etiquette'] == 'spam').to_numpy(int)
X6, y6 = six[['liens', 'mots_suspects']].to_numpy(float), (six['etiquette'] == 'spam').to_numpy(int)
filtres = {'A': ((1, 0.5), -4), 'B': ((-0.54, 4.39), -9.17)}
for nom, (w, b) in filtres.items():
    p6 = sigmoide(X6 @ np.array(w) + b)
    pertes6 = -(y6 * np.log(p6) + (1 - y6) * np.log(1 - p6))
    print(f'filtre {nom} : entraînement {perte_moyenne(y12, X12, w, b):.2f}, validation {perte_moyenne(y6, X6, w, b):.2f} '
          f'| pertes des six : {np.round(pertes6, 2)}')

# Perte pénalisée L2 (équation 6.2), λ = 0,05 ; la constante b n'est pas pénalisée
lam = 0.05
for nom, (w, b) in filtres.items():
    pen = lam * np.sum(np.square(w))
    print(f'filtre {nom} : somme des carrés des poids = {np.sum(np.square(w)):.4f}, pénalité = {pen:.4f}, '
          f'perte pénalisée = {perte_moyenne(y12, X12, w, b) + pen:.2f}')

# La pénalité dépend des unités : liens comptés en dizaines, mêmes prédictions, autre pénalité
X12_diz = X12 / np.array([10, 1])
w_diz = (10, 0.5)
print('mêmes probabilités :', np.allclose(sigmoide(X12_diz @ np.array(w_diz) - 4), sigmoide(X12 @ np.array([1, 0.5]) - 4)),
      f'| pénalité en unités : {lam * (1 + 0.25):.4f} ; en dizaines : {lam * (100 + 0.25):.4f}')

In [ ]:
# Leçon 6.4 — standardisation dans chaque tour : moyenne et écart-type des liens sur les DEUX AUTRES plis
for k in range(3):
    autres = plis[plis['pli'] != k + 1]
    n = autres['messages'].sum()
    m = (autres['messages'] * autres['moyenne_liens']).sum() / n
    # écart-type de la réunion de deux plis (division par n) : moyenne des carrés, moins le carré de la moyenne
    v = (autres['messages'] * (autres['ecart_type_liens'] ** 2 + autres['moyenne_liens'] ** 2)).sum() / n - m ** 2
    print(f'tour {k + 1} : validation = pli {k + 1}, entraînement = {n} messages, moyenne {m:.2f}, écart-type {np.sqrt(v):.2f}')
print(f'message à 6 liens du pli 3, au tour 3 : (6 - 2,25) / 1,76 = {(6 - 2.25) / 1.76:.2f}')

# Score de validation croisée (équation 6.4) des deux valeurs de λ
for col, lam in (('perte_l001', 0.01), ('perte_l01', 0.1)):
    print(f'λ = {lam} : pertes des plis {list(plis[col])}, VC = {plis[col].mean():.3f}')

In [ ]:
# Leçon 6.4 (fin) — L1 et L2 sur des messages fictifs à huit caractéristiques (les mêmes qu'à l'exercice 6.4)
msgs = fabriquer_messages()
X8 = msgs[NOMS_CARAC].to_numpy(float)
X8 = (X8 - X8.mean(0)) / X8.std(0)        # ici, standardisation sur tous les messages : simple démonstration
y8 = msgs['spam'].to_numpy(float)
w_l2, _ = ajuster(X8, y8, 0.1)
w_l1, _ = ajuster(X8, y8, 0.1, l1=True)
print('poids L2 :', np.round(w_l2, 2))
print('poids L1 :', np.round(w_l1, 2))
print(f'poids nuls : L2 {int(np.sum(np.abs(w_l2) < 1e-9))}, L1 {int(np.sum(np.abs(w_l1) < 1e-9))} '
      '(les deux premières caractéristiques sont les seules liées à l étiquette)')

In [ ]:
# Leçons 6.5 et 6.6 — le test A/B de la librairie
a, b_ = librairie.iloc[0], librairie.iloc[1]
pA, pB = a['commandes'] / a['clients'], b_['commandes'] / b_['clients']
ecart = pB - pA
print(f'taux A = {100 * pA:.1f} %, taux B = {100 * pB:.1f} %, écart = {100 * ecart:.1f} points, '
      f'variation relative = {100 * ecart / pA:.0f} %')
esA, esB = np.sqrt(pA * (1 - pA) / a['clients']), np.sqrt(pB * (1 - pB) / b_['clients'])
es = np.sqrt(esA ** 2 + esB ** 2)                          # équation 6.6 : on additionne les CARRÉS
bas, haut = ecart - 1.96 * es, ecart + 1.96 * es           # équation 6.7
print(f'ES de A = {100 * esA:.2f} point, ES de B = {100 * esB:.2f} point, ES de l écart = {100 * es:.2f} point')
print(f'intervalle de confiance à 95 % : de {100 * bas:.1f} à {100 * haut:.1f} points')
print(f'pour 40 000 abonnés : {40000 * ecart:.0f} commandes de plus (de {40000 * bas:.0f} à {40000 * haut:.0f})')

plt.figure(figsize=(6, 1.8))
plt.axvline(0, color='black', linewidth=2.5, label='0 : aucun effet')
plt.axvline(1, color=SEUIL, linestyle='--', label='seuil de rentabilité : 1 point')
plt.plot([100 * bas, 100 * haut], [0, 0], color=ECART, linewidth=4, label='intervalle à 95 %')
plt.scatter([100 * ecart], [0], marker='D', s=70, color=ECART, zorder=3)
plt.yticks([]); plt.xlim(-1, 5); plt.xlabel('Effet de la version B (points de pourcentage)')
plt.legend(fontsize=7, loc='upper right'); plt.show()

# Leçon 6.6 — le coût par message du test (leçon 6.1) est une moyenne : même raisonnement
couts = np.array([1] * 20 + [3] * 4 + [0] * 226)            # 20 spams manqués, 4 fausses alertes, 226 sans coût
es_c = couts.std() / np.sqrt(len(couts))                     # écart-type (division par n) / racine de n
print(f'coût par message = {couts.mean():.3f}, écart-type = {couts.std():.3f}, ES = {es_c:.3f}, '
      f'intervalle : de {couts.mean() - 1.96 * es_c:.2f} à {couts.mean() + 1.96 * es_c:.2f}')

## Exercice 6.1 — Lire un graphique : trois plis, cinq valeurs de λ

*Notions : validation croisée (leçon 6.4, équation 6.4), surapprentissage et pénalité (leçon 6.3).* Données D6.E1 : la perte de validation de chaque pli de la médiathèque, pour cinq valeurs de $\lambda$.

### a) à e) Sur papier, en lisant la figure du livre

*Vos lectures et vos réponses :* …

In [ ]:
# À COMPLÉTER : moyenne des trois plis pour chaque λ (équation 6.4), puis le λ choisi
moyennes = None   # À COMPLÉTER : une Series indexée par λ ; essayez plis_lambda.groupby('lambda')['perte']
lam_choisi = None   # À COMPLÉTER

In [ ]:
# Vérification de a) et b)
if moyennes is None or lam_choisi is None:
    print('pas encore complété. Complétez les deux lignes.')
else:
    print('moyennes :', moyennes.round(3).to_dict(), '→',
          'correct' if np.allclose(moyennes.values, [0.51, 1.3 / 3, 1.09 / 3, 1.1 / 3, 1.58 / 3]) else 'à revoir')
    print('λ choisi :', lam_choisi, '→', 'correct' if lam_choisi == 0.1 else 'à revoir')
    print('choix de chaque pli seul :', plis_lambda.loc[plis_lambda.groupby('pli')['perte'].idxmin(), ['pli', 'lambda']].values.tolist())
    for k, (style, marque) in enumerate((('-', 'o'), ('--', 's'), (':', '^'))):
        d = plis_lambda[plis_lambda['pli'] == k + 1]
        plt.plot(d['lambda'], d['perte'], style, marker=marque, color=LOT_V, label=f'pli {k + 1}')
    plt.plot(moyennes.index, moyennes.values, 'D-', color='black', linewidth=2.5, label='moyenne')
    plt.xscale('log'); plt.xlabel('λ'); plt.ylabel('Perte de validation'); plt.legend(); plt.grid(alpha=0.4); plt.show()

## Exercice 6.2 — Choisir, puis évaluer un seuil : la médiathèque

*Notions : trois lots (leçon 6.1), coût par message (équation 6.1), coût d'un seuil (équation 5.9).* Pour la médiathèque, une fausse alerte coûte deux spams manqués : $k = 2$. Données D6.E2. Faites d'abord les questions a) à e) sur papier, **sans regarder les lignes du test avant la question b)**.

### a) à e) Sur papier

*Vos calculs et vos réponses :* …

In [ ]:
# À COMPLÉTER : coût par message (équation 6.1) et choix du seuil sur la VALIDATION
def cout_par_message(fn, fp, n, k):
    return None   # À COMPLÉTER


def choisir_seuil(table_validation, k):
    """Renvoie le seuil dont le coût par message est le plus petit sur la validation."""
    return None   # À COMPLÉTER

In [ ]:
# Vérification de a) et b), puis le protocole
k_med = 2
valid = mediatheque[mediatheque['lot'] == 'validation']
test_med = mediatheque[mediatheque['lot'] == 'test']
if cout_par_message(3, 18, 150, 2) is None or choisir_seuil(valid, k_med) is None:
    print('pas encore complété. Complétez les fonctions.')
else:
    cv = [cout_par_message(l.fn, l.fp, l.messages, k_med) for l in valid.itertuples()]
    print('validation, coût par message :', np.round(cv, 3), '→',
          'correct' if np.allclose(cv, [39 / 150, 24 / 150, 20 / 150, 21 / 150]) else 'à revoir')
    s = choisir_seuil(valid, k_med)
    print(f'seuil choisi sur la validation : {s} →', 'correct' if s == 0.6 else 'à revoir')
    l = test_med[test_med['seuil'] == s].iloc[0]                    # UNE seule ligne du test
    ct = cout_par_message(l.fn, l.fp, l.messages, k_med)
    print(f'test, au seuil {s} seulement : {ct:.3f} par message →', 'correct' if abs(ct - 35 / 240) < 1e-9 else 'à revoir')
    # e) Ce que fait le collègue : il regarde TOUTES les lignes du test et garde la meilleure
    ctous = [cout_par_message(r.fn, r.fp, r.messages, k_med) for r in test_med.itertuples()]
    s_coll = test_med['seuil'].iloc[int(np.argmin(ctous))]
    print(f'annonce du collègue : seuil {s_coll}, {min(ctous):.3f} par message (choisi sur le test : trop optimiste) ;',
          f'annonce honnête : seuil {s}, {ct:.3f} par message →',
          'correct' if s_coll == 0.7 and abs(min(ctous) - 28 / 240) < 1e-9 else 'à revoir')
    plt.figure(figsize=(5.5, 3))
    plt.plot(valid['seuil'], cv, 'o-', color=LOT_V, label='validation (sert à choisir)')
    plt.plot(test_med['seuil'], ctous, 's--', color=LOT_T, alpha=0.5, label='test (à ne pas regarder pour choisir)')
    plt.scatter([s], [ct], s=120, color=LOT_T, zorder=3, label='test, au seuil choisi')
    plt.xlabel('Seuil'); plt.ylabel('Coût par message (k = 2)'); plt.legend(fontsize=8); plt.grid(alpha=0.4); plt.show()

### f) Votre conclusion

*Ce que la médiathèque peut annoncer :* …

## Exercice 6.3 — Interpréter : une fuite entre les plis

*Notions : fuite de données (leçon 6.2), validation croisée par groupes et standardisation dans chaque tour (leçon 6.4).* Données D6.E3 : où sont tombés les messages de quatre groupes dans les plis du bénévole.

### a) à d) Sur papier

*Vos réponses :* …

In [ ]:
# À COMPLÉTER : groupes présents dans plus d'un pli
def groupes_coupes(repartition):
    """repartition : un tableau avec une colonne 'groupe' et les colonnes 'pli_1', 'pli_2', 'pli_3'
    (nombre de messages du groupe dans chaque pli). Renvoie la liste des groupes coupés."""
    return None   # À COMPLÉTER : comptez, pour chaque groupe, les plis où il a au moins un message

In [ ]:
# Vérification de b) et d)
proposition = pd.DataFrame([('Montres', 26, 0, 0), ('Colis en attente', 0, 0, 15),
                            ('Loterie', 0, 22, 0), ('Lettre de la mairie', 0, 0, 12)],
                           columns=['groupe', 'pli_1', 'pli_2', 'pli_3'])
if groupes_coupes(groupes_plis) is None:
    print('pas encore complété. Complétez la fonction.')
else:
    print('plis du bénévole, groupes coupés :', groupes_coupes(groupes_plis), '→',
          'correct' if len(groupes_coupes(groupes_plis)) == 4 else 'à revoir')
    print('répartition corrigée, groupes coupés :', groupes_coupes(proposition), '| tailles des plis :',
          proposition[['pli_1', 'pli_2', 'pli_3']].sum().tolist(), '→',
          'correct' if groupes_coupes(proposition) == [] else 'à revoir')
    print('tour 1 : copies « Montres » vues à l entraînement =', int(groupes_plis.loc[0, ['pli_2', 'pli_3']].sum()),
          ', en validation =', int(groupes_plis.loc[0, 'pli_1']))

## Exercice 6.4 — Validation croisée par campagnes

*Notions : validation croisée (leçon 6.4, équation 6.4), pénalité L2 (équation 6.2), standardisation (équation 2.7), fuite (leçon 6.2).* Les messages sont fabriqués par `fabriquer_messages()` (cellule d'import). Les groupes 30 à 39 forment le lot de test : on n'y touche qu'à la question e).

In [ ]:
# Les messages de l'exercice : 30 groupes pour la validation croisée, 10 groupes mis de côté pour le test
msgs = fabriquer_messages()
test_ = msgs['groupe'] >= 30
X_cv, y_cv, g_cv = msgs.loc[~test_, NOMS_CARAC].to_numpy(float), msgs.loc[~test_, 'spam'].to_numpy(float), msgs.loc[~test_, 'groupe'].to_numpy()
X_test, y_test = msgs.loc[test_, NOMS_CARAC].to_numpy(float), msgs.loc[test_, 'spam'].to_numpy(float)
print(f'{len(msgs)} messages, {msgs.groupe.nunique()} groupes ; validation croisée : {len(y_cv)} messages ; test : {len(y_test)} messages')
print(msgs[NOMS_CARAC].describe().loc[['mean', 'std']].round(1))     # des unités très différentes

### a) Votre prédiction

*Quand λ passe de 0 à 1, la perte de validation croisée…* …

In [ ]:
# À COMPLÉTER : b) plis par groupe, standardisation sur l'entraînement, c) score de validation croisée
def plis_par_groupe(groupes, K):
    """Numéro de pli (0 à K - 1) de chaque message : les groupes, rangés par ordre croissant, sont distribués
    à tour de rôle dans les plis ; tous les messages d'un groupe reçoivent le même pli."""
    return None   # À COMPLÉTER ; np.unique(groupes) donne les groupes rangés


def standardiser(X_entr, X_val):
    """Standardise X_entr et X_val avec la moyenne et l'écart-type de X_entr SEULEMENT."""
    return None, None   # À COMPLÉTER


def score_vc(lam, plis_, K=3):
    """Équation 6.4 : moyenne, sur les K tours, de la perte SANS pénalité sur le pli de validation."""
    return None   # À COMPLÉTER : utilisez standardiser, ajuster(X, y, lam) et perte_moyenne(y, X, w, b)

In [ ]:
# Vérification de b) et c)
LAMBDAS = [0, 0.001, 0.01, 0.1, 1]
p_g = plis_par_groupe(g_cv, 3)
if p_g is None or standardiser(X_cv, X_cv)[0] is None or score_vc(0.1, p_g) is None:
    print('pas encore complété. Complétez les trois fonctions.')
else:
    coupes = sum(len(set(p_g[g_cv == g])) > 1 for g in np.unique(g_cv))
    print(f'plis par groupe : {np.bincount(p_g).tolist()} messages, {coupes} groupe(s) coupé(s) →',
          'correct' if coupes == 0 and set(p_g) == {0, 1, 2} else 'à revoir')
    Xe, Xv = standardiser(X_cv[p_g != 0], X_cv[p_g == 0])
    print('entraînement standardisé : moyennes ≈ 0 et écarts-types ≈ 1 ; validation : pas exactement →',
          'correct' if np.allclose(Xe.mean(0), 0) and np.allclose(Xe.std(0), 1) and not np.allclose(Xv.mean(0), 0) else 'à revoir')
    vc_groupe = {lam: score_vc(lam, p_g) for lam in LAMBDAS}
    lam_vc = min(vc_groupe, key=vc_groupe.get)
    print('VC par groupe :', {l: round(v, 3) for l, v in vc_groupe.items()}, '→ λ choisi :', lam_vc, '→',
          'correct' if lam_vc == 0.01 else 'à revoir')

In [ ]:
# d) Plis tirés message par message (à ne pas faire) : la fuite de la leçon 6.2, trois fois
if p_g is not None and score_vc(0.1, p_g) is not None:
    p_m = np.random.default_rng(1).integers(0, 3, len(y_cv))
    vc_message = {lam: score_vc(lam, p_m) for lam in LAMBDAS}
    print('VC message par message :', {l: round(v, 3) for l, v in vc_message.items()})
    plt.plot(range(5), list(vc_groupe.values()), 'o-', color=LOT_V, label='plis par groupe')
    plt.plot(range(5), list(vc_message.values()), 's--', color='grey', label='plis message par message (fuite)')
    plt.xticks(range(5), [str(l) for l in LAMBDAS]); plt.xlabel('λ'); plt.ylabel('Score de validation croisée')
    plt.legend(); plt.grid(alpha=0.4); plt.show()

    # e) Modèle final : standardisation et poids sur les 30 groupes, avec le λ choisi ; test UNE seule fois
    Xc, Xt = standardiser(X_cv, X_test)
    w_f, b_f = ajuster(Xc, y_cv, lam_vc)
    print(f'perte de test du modèle final (λ = {lam_vc}) : {perte_moyenne(y_test, Xt, w_f, b_f):.3f} ; '
          f'VC par groupe : {vc_groupe[lam_vc]:.3f} ; VC message par message : {vc_message[lam_vc]:.3f}')

### d) et e) Votre interprétation

…

## Exercice 6.5 — Prédire, puis vérifier : la relance du théâtre

*Notions : écart en points (équation 6.5), erreur standard (équation 6.6), intervalle de confiance (équation 6.7).* Données D6.E5. La vidéo n'est utile que si elle fait gagner au moins 0,5 point.

### a) à e) Sur papier

*Vos calculs, votre prédiction et vos réponses :* …

In [ ]:
# À COMPLÉTER : écart en points et intervalle de confiance à 95 % (équations 6.5 à 6.7)
def intervalle(conv_A, n_A, conv_B, n_B):
    """Renvoie (écart, borne basse, borne haute), en points de pourcentage."""
    return None   # À COMPLÉTER

In [ ]:
# Vérification de a), b) et e)
A, B = theatre.iloc[0], theatre.iloc[1]
r = intervalle(A['reservations'], A['spectateurs'], B['reservations'], B['spectateurs'])
if r is None:
    print('pas encore complété. Complétez la fonction.')
else:
    print(f'essai réel : écart {r[0]:.2f} points, intervalle de {r[1]:.2f} à {r[2]:.2f} →',
          'correct' if abs(r[0] - 1.4) < 1e-9 and abs(r[1] + 0.35) < 0.01 and abs(r[2] - 3.15) < 0.01 else 'à revoir')
    r4 = intervalle(4 * A['reservations'], 4 * A['spectateurs'], 4 * B['reservations'], 4 * B['spectateurs'])
    print(f'essai quatre fois plus grand : de {r4[1]:.2f} à {r4[2]:.2f} ; largeur {r[2] - r[1]:.2f} → {r4[2] - r4[1]:.2f} →',
          'correct' if abs((r[2] - r[1]) / (r4[2] - r4[1]) - 2) < 1e-9 and r4[1] > 0.5 else 'à revoir')

In [ ]:
# f) 1 000 essais simulés dont le véritable effet vaut 1,4 point : combien d'intervalles contiennent 1,4 ?
rng = np.random.default_rng(65)
nA, nB, vrai_A, vrai_B = 1800, 1750, 0.07, 0.084
a = rng.binomial(nA, vrai_A, 1000) / nA
b = rng.binomial(nB, vrai_B, 1000) / nB
es_sim = np.sqrt(a * (1 - a) / nA + b * (1 - b) / nB)
contient = np.abs((b - a) - 0.014) <= 1.96 * es_sim
print(f'part des intervalles qui contiennent le véritable effet : {contient.mean():.1%}')
print(f'part des intervalles qui contiennent zéro : {np.mean(np.abs(b - a) <= 1.96 * es_sim):.1%}')

### f) Votre conclusion

…

## Exercice 6.6 — Défi : combien de clients, et quand regarder ?

*Notions : erreur standard et intervalle de confiance (leçon 6.6), loi binomiale (leçon 4.5).*

### a) à c) Sur papier

*Vos calculs et votre prédiction :* …

In [ ]:
# À COMPLÉTER : a) taille par groupe pour une demi-largeur donnée (en points)
def taille_par_groupe(pA, pB, demi_largeur):
    """pA, pB en pourcentage ; renvoie le plus petit n entier tel que 1,96 × ES ≤ demi_largeur."""
    return None   # À COMPLÉTER ; np.ceil arrondit à l'entier supérieur

In [ ]:
# Vérification de a) et b)
n1 = taille_par_groupe(8, 10, 1)
if n1 is None:
    print('pas encore complété. Complétez la fonction.')
else:
    print(f'demi-largeur 1 point : {n1} clients par groupe →', 'correct' if n1 == 6285 else 'à revoir')
    print(f'demi-largeur 0,5 point : {taille_par_groupe(8, 10, 0.5)} clients par groupe, '
          f'{2 * taille_par_groupe(8, 10, 0.5)} en tout')

In [ ]:
# d) 2 000 essais SANS effet. Calendrier : 250 clients par jour et par groupe, 20 lectures (une chaque soir,
#    sur les clients cumulés : 250, 500… 5 000 par groupe). Règle d'arrêt : premier soir où l'intervalle exclut 0.
rng = np.random.default_rng(66)
essais, jours, par_jour, p = 2000, 20, 250, 0.08
cA = rng.binomial(par_jour, p, size=(essais, jours)).cumsum(axis=1)     # commandes cumulées, groupe A
cB = rng.binomial(par_jour, p, size=(essais, jours)).cumsum(axis=1)     # groupe B : même taux, aucun effet
n = par_jour * np.arange(1, jours + 1)                                  # clients par groupe à la fin de chaque jour
pA_, pB_ = cA / n, cB / n
es_j = np.sqrt(pA_ * (1 - pA_) / n + pB_ * (1 - pB_) / n)
exclut = np.abs(pB_ - pA_) > 1.96 * es_j                                # l'intervalle du soir exclut-il zéro ?
fin, un_soir = exclut[:, -1].mean(), exclut.any(axis=1).mean()
print(f'intervalle lu seulement le 20e jour : {fin:.1%} des essais concluent à tort à un effet')
print(f'intervalle lu chaque soir, arrêt au premier « succès » : {un_soir:.1%}')

### d) et e) Votre explication

…